In [ ]:
# Cell 1 – Import ALL required libraries
import os, time, random, warnings

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

from tensorflow.keras.applications import MobileNet, MobileNetV2, MobileNetV3Small
from tensorflow.keras import layers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

print('TensorFlow :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

: 

In [ ]:
# Cell 2 – Download CIFAR-10 & split 70 / 15 / 15
DATA_DIR = './cifar10_data'
os.makedirs(DATA_DIR, exist_ok=True)

(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.cifar10.load_data()

X = np.concatenate([x_tr, x_te], axis=0).astype('float32') / 255.0
y = np.concatenate([y_tr, y_te], axis=0).reshape(-1)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# 70 – 15 – 15 voi stratify
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

print(f'Train : {X_train.shape}  ({len(X_train)/len(X)*100:.1f} %)')
print(f'Val   : {X_val.shape}    ({len(X_val)/len(X)*100:.1f} %)')
print(f'Test  : {X_test.shape}   ({len(X_test)/len(X)*100:.1f} %)')

# Preview 10 anh dau
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i])
    ax.set_title(CLASS_NAMES[y_train[i]])
    ax.axis('off')
plt.suptitle('CIFAR-10 samples', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# Library : tensorflow.keras.applications.MobileNet  <- V1
# Class   : TF_MobileNetV1
# ==============================================================================
class TF_MobileNetV1:
    '''
    MobileNetV1 - depthwise separable convolutions (Howard et al., 2017).

    [Tho may AI] Chinh sua lop dau vao:
        CIFAR-10 32x32  --layers.Resizing-->  224x224  --> MobileNetV1 backbone
    Backbone dong bang (ImageNet weights), chi train custom head.
    '''

    def __init__(self, num_classes=10, input_shape=(32, 32, 3),
                 epochs=1000, batch_size=64):
        self.num_classes   = num_classes
        self.input_shape   = input_shape
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.train_time    = 0

        # ── 5 history buffers ────────────────────────────────────────
        self.loss          = []
        self.val_loss      = []
        self.accuracy      = []
        self.val_accuracy  = []
        self.lr_history    = []

        self.model = self._build_model()
        tp = sum(tf.size(v).numpy() for v in self.model.trainable_variables)
        print(f'[TF_MobileNetV1] Total params     : {self.model.count_params():,}')
        print(f'[TF_MobileNetV1] Trainable params : {tp:,}')

    def _build_model(self):
        '''
        [CORE - Tho may AI] Chinh sua lop dau vao:
        Them layers.Resizing(224, 224) vao dau model thay vi resize anh ben ngoai.
        '''
        inputs = tf.keras.Input(shape=self.input_shape, name='input_32x32')

        # [CORE] Chinh sua lop dau vao: 32x32 --> 224x224
        x = layers.Resizing(224, 224, name='resize_32_to_224')(inputs)
        # Rescale [0,1] --> [0,255] roi ap preprocessing chuan MobileNetV1
        x = layers.Rescaling(255.0, name='rescale_to_255')(x)
        x = layers.Lambda(
            lambda t: tf.keras.applications.mobilenet.preprocess_input(t),
            name='preprocess_v1'
        )(x)

        # Pretrained backbone (frozen)
        backbone = MobileNet(
            weights='imagenet', include_top=False,
            input_shape=(224, 224, 3), alpha=1.0,
        )
        backbone.trainable = False
        x = backbone(x, training=False)

        # Custom classification head
        x = layers.GlobalAveragePooling2D(name='gap')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(
            self.num_classes, activation='softmax', name='predictions'
        )(x)

        model = tf.keras.Model(inputs, outputs, name='TF_MobileNetV1')
        model.compile(
            optimizer=optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def train(self, X_train, y_train, X_val, y_val):
        cbs = [
            EarlyStopping(monitor='val_accuracy', patience=25,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=10, min_lr=1e-7, verbose=1),
        ]
        t0 = time.time()
        hist = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=cbs, verbose=1,
        )
        self.train_time   = time.time() - t0
        self.loss         = hist.history['loss']
        self.val_loss     = hist.history['val_loss']
        self.accuracy     = hist.history['accuracy']
        self.val_accuracy = hist.history['val_accuracy']
        self.lr_history   = hist.history.get('lr', [])
        print(f'\n V1 - {len(self.loss)} epochs | {self.train_time:.1f}s')
        return hist

    def forward(self, X):
        '''Inference - tra ve mang xac suat (batch, num_classes).'''
        return self.model.predict(X, verbose=0)

In [ ]:
# ==============================================================================
# Library : tensorflow.keras.applications.MobileNetV2  <- V2
# Class   : TF_MobileNetV2
# ==============================================================================
class TF_MobileNetV2:
    '''
    MobileNetV2 - inverted residuals + linear bottlenecks (Sandler et al., 2018).

    [Tho may AI] Chinh sua lop dau vao:
        CIFAR-10 32x32  --layers.Resizing-->  224x224  --> MobileNetV2 backbone
    '''

    def __init__(self, num_classes=10, input_shape=(32, 32, 3),
                 epochs=1000, batch_size=64):
        self.num_classes   = num_classes
        self.input_shape   = input_shape
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.train_time    = 0

        self.loss          = []
        self.val_loss      = []
        self.accuracy      = []
        self.val_accuracy  = []
        self.lr_history    = []

        self.model = self._build_model()
        tp = sum(tf.size(v).numpy() for v in self.model.trainable_variables)
        print(f'[TF_MobileNetV2] Total params     : {self.model.count_params():,}')
        print(f'[TF_MobileNetV2] Trainable params : {tp:,}')

    def _build_model(self):
        inputs = tf.keras.Input(shape=self.input_shape, name='input_32x32')

        # [CORE] Chinh sua lop dau vao: 32x32 --> 224x224
        x = layers.Resizing(224, 224, name='resize_32_to_224')(inputs)
        x = layers.Rescaling(255.0, name='rescale_to_255')(x)
        x = layers.Lambda(
            lambda t: tf.keras.applications.mobilenet_v2.preprocess_input(t),
            name='preprocess_v2'
        )(x)

        backbone = MobileNetV2(
            weights='imagenet', include_top=False,
            input_shape=(224, 224, 3), alpha=1.0,
        )
        backbone.trainable = False
        x = backbone(x, training=False)

        x = layers.GlobalAveragePooling2D(name='gap')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(
            self.num_classes, activation='softmax', name='predictions'
        )(x)

        model = tf.keras.Model(inputs, outputs, name='TF_MobileNetV2')
        model.compile(
            optimizer=optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def train(self, X_train, y_train, X_val, y_val):
        cbs = [
            EarlyStopping(monitor='val_accuracy', patience=25,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=10, min_lr=1e-7, verbose=1),
        ]
        t0 = time.time()
        hist = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=cbs, verbose=1,
        )
        self.train_time   = time.time() - t0
        self.loss         = hist.history['loss']
        self.val_loss     = hist.history['val_loss']
        self.accuracy     = hist.history['accuracy']
        self.val_accuracy = hist.history['val_accuracy']
        self.lr_history   = hist.history.get('lr', [])
        print(f'\n V2 - {len(self.loss)} epochs | {self.train_time:.1f}s')
        return hist

    def forward(self, X):
        return self.model.predict(X, verbose=0)

In [ ]:
# ==============================================================================
# Library : tensorflow.keras.applications.MobileNetV3Small  <- V3
# Class   : TF_MobileNetV3Small
# ==============================================================================
class TF_MobileNetV3Small:
    '''
    MobileNetV3-Small - SE blocks + h-swish, NAS-optimized (Howard et al., 2019).

    [Tho may AI] Chinh sua lop dau vao:
        CIFAR-10 32x32  --layers.Resizing-->  224x224  --> MobileNetV3Small backbone
    '''

    def __init__(self, num_classes=10, input_shape=(32, 32, 3),
                 epochs=1000, batch_size=64):
        self.num_classes   = num_classes
        self.input_shape   = input_shape
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.train_time    = 0

        self.loss          = []
        self.val_loss      = []
        self.accuracy      = []
        self.val_accuracy  = []
        self.lr_history    = []

        self.model = self._build_model()
        tp = sum(tf.size(v).numpy() for v in self.model.trainable_variables)
        print(f'[TF_MobileNetV3Small] Total params     : {self.model.count_params():,}')
        print(f'[TF_MobileNetV3Small] Trainable params : {tp:,}')

    def _build_model(self):
        inputs = tf.keras.Input(shape=self.input_shape, name='input_32x32')

        # [CORE] Chinh sua lop dau vao: 32x32 --> 224x224
        x = layers.Resizing(224, 224, name='resize_32_to_224')(inputs)
        # V3 preprocess: [0,255] --> [-1, 1]
        x = layers.Rescaling(255.0, name='rescale_to_255')(x)
        x = layers.Lambda(
            lambda t: tf.keras.applications.mobilenet_v3.preprocess_input(t),
            name='preprocess_v3'
        )(x)

        backbone = MobileNetV3Small(
            weights='imagenet', include_top=False,
            input_shape=(224, 224, 3),
        )
        backbone.trainable = False
        x = backbone(x, training=False)

        x = layers.GlobalAveragePooling2D(name='gap')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.2)(x)
        outputs = layers.Dense(
            self.num_classes, activation='softmax', name='predictions'
        )(x)

        model = tf.keras.Model(inputs, outputs, name='TF_MobileNetV3Small')
        model.compile(
            optimizer=optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'],
        )
        return model

    def train(self, X_train, y_train, X_val, y_val):
        cbs = [
            EarlyStopping(monitor='val_accuracy', patience=25,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=10, min_lr=1e-7, verbose=1),
        ]
        t0 = time.time()
        hist = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=cbs, verbose=1,
        )
        self.train_time   = time.time() - t0
        self.loss         = hist.history['loss']
        self.val_loss     = hist.history['val_loss']
        self.accuracy     = hist.history['accuracy']
        self.val_accuracy = hist.history['val_accuracy']
        self.lr_history   = hist.history.get('lr', [])
        print(f'\n V3-Small - {len(self.loss)} epochs | {self.train_time:.1f}s')
        return hist

    def forward(self, X):
        return self.model.predict(X, verbose=0)

In [ ]:
# ==============================================================================
# Khoi tao & train ca 3 model – so sanh tren cung tap du lieu
# ==============================================================================

# Instantiate
v1 = TF_MobileNetV1()
v2 = TF_MobileNetV2()
v3 = TF_MobileNetV3Small()

MODEL_INFO = [
    ('MobileNetV1',       v1, '#E74C3C'),
    ('MobileNetV2',       v2, '#3498DB'),
    ('MobileNetV3-Small', v3, '#2ECC71'),
]

# Train
for name, m, _ in MODEL_INFO:
    print('\n' + '=' * 65)
    print(f'  Training {name} ...')
    print('=' * 65)
    m.train(X_train, y_train, X_val, y_val)

# Evaluate on test set
print('\n' + '=' * 65)
print('  TEST SET RESULTS')
print('=' * 65)

results = {}
for name, m, color in MODEL_INFO:
    prob  = m.forward(X_test)
    y_hat = prob.argmax(axis=1)
    acc   = accuracy_score(y_test, y_hat)
    results[name] = {
        'accuracy':         acc,
        'train_time':       m.train_time,
        'total_params':     m.model.count_params(),
        'trainable_params': sum(tf.size(v).numpy() for v in m.model.trainable_variables),
        'epochs_run':       len(m.loss),
        'y_pred':           y_hat,
        'color':            color,
    }
    print(f'  {name:<22} | acc={acc:.4f} | time={m.train_time:>7.1f}s '
          f'| epochs={len(m.loss)}')

In [ ]:
# ==============================================================================
# Bieu do tong hop so sanh 3 mo hinh
# ==============================================================================
model_objs = [v1, v2, v3]
names      = [n for n, *_ in MODEL_INFO]
colors     = [c for _, __, c in MODEL_INFO]

# ─────────────────────────────────────────────────────────────────────────────
# 1) Loss & Accuracy curves
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training Curves - MobileNet V1 / V2 / V3', fontsize=16, fontweight='bold')
(ax_tl, ax_tr), (ax_bl, ax_br) = axes

for m, name, color in zip(model_objs, names, colors):
    ep = range(1, len(m.loss) + 1)
    ax_tl.plot(ep, m.loss,         color=color, lw=2,                 label=name)
    ax_tr.plot(ep, m.val_loss,     color=color, lw=2,                 label=name)
    ax_bl.plot(ep, m.accuracy,     color=color, lw=2,                 label=name)
    ax_br.plot(ep, m.val_accuracy, color=color, lw=2, linestyle='--', label=name)

for ax, title, ylabel in [
    (ax_tl, 'Train Loss',     'Loss'),
    (ax_tr, 'Val Loss',       'Loss'),
    (ax_bl, 'Train Accuracy', 'Accuracy'),
    (ax_br, 'Val Accuracy',   'Accuracy'),
]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 2) Bar charts – Accuracy / Time / Params / Epochs
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Model Comparison – Test Set', fontsize=15, fontweight='bold')

metrics = {
    'Test Accuracy':    [results[n]['accuracy']          for n in names],
    'Train Time (s)':   [results[n]['train_time']         for n in names],
    'Total Params (M)': [results[n]['total_params'] / 1e6 for n in names],
    'Epochs Run':       [results[n]['epochs_run']          for n in names],
}
for ax, (title, vals) in zip(axes, metrics.items()):
    bars = ax.bar(names, vals, color=colors, edgecolor='black', linewidth=0.8, width=0.55)
    ax.set_title(title, fontweight='bold'); ax.set_ylabel(title)
    ax.set_xticklabels(names, rotation=15, ha='right')
    for bar, v in zip(bars, vals):
        fmt = f'{v:.3f}' if isinstance(v, float) else str(int(v))
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                fmt, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 3) Confusion matrices
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=15, fontweight='bold')

for ax, (name, m, _) in zip(axes, MODEL_INFO):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, cbar=False)
    acc_val = results[name]['accuracy']
    ax.set_title(f'{name}\nAcc = {acc_val:.4f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 4) Per-class F1-score heatmap  (correlation-style matrix)
# ─────────────────────────────────────────────────────────────────────────────
f1_data = {}
for name, m, _ in MODEL_INFO:
    _, _, f1, _ = precision_recall_fscore_support(
        y_test, results[name]['y_pred'], average=None, labels=list(range(10)))
    f1_data[name] = f1

f1_df = pd.DataFrame(f1_data, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(f1_df, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Per-class F1-score Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Model'); ax.set_ylabel('Class')
plt.tight_layout(); plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 5) Classification reports
# ─────────────────────────────────────────────────────────────────────────────
for name, m, _ in MODEL_INFO:
    print('=' * 65)
    print(f'  Classification Report - {name}')
    print('=' * 65)
    print(classification_report(y_test, results[name]['y_pred'],
                                 target_names=CLASS_NAMES))

# ─────────────────────────────────────────────────────────────────────────────
# 6) Summary table
# ─────────────────────────────────────────────────────────────────────────────
# FLOPs ly thuyet cho backbone 224x224 (tu paper goc)
THEORETICAL_FLOPS_M = {
    'MobileNetV1':        569,
    'MobileNetV2':        300,
    'MobileNetV3-Small':   56,
}

rows = []
for name, m, _ in MODEL_INFO:
    r   = results[name]
    tp  = r['total_params']
    trp = r['trainable_params']
    acc = r['accuracy']
    t   = r['train_time']
    ep  = r['epochs_run']
    rows.append({
        'Model':            name,
        'Total Params':     f'{tp:,}',
        'Trainable Params': f'{trp:,}',
        'FLOPs (M)':        THEORETICAL_FLOPS_M.get(name, '-'),
        'Test Accuracy':    f'{acc:.4f}',
        'Train Time (s)':   f'{t:.1f}',
        'Epochs Run':       ep,
    })

summary = pd.DataFrame(rows).set_index('Model')
print('\n' + '=' * 70)
print('  SUMMARY TABLE')
print('=' * 70)
print(summary.to_string())
print('\n* FLOPs: theoretical backbone FLOPs at 224x224 (from papers)')
display(
    summary.style
    .highlight_max(subset=['Test Accuracy'], color='lightgreen')
    .highlight_min(subset=['Train Time (s)'], color='lightyellow')
)

In [ ]:
# ==============================================================================
# KET LUAN
# ==============================================================================
print('''
+------------------------------------------------------------------+
|        KET LUAN – MobileNet V1 / V2 / V3-Small  (CIFAR-10)      |
+------------------+-----------------------------------------------+
| MobileNetV1      | Params lon nhat; kien truc don gian, de debug  |
|                  | Phu hop: phan cung cu, can simplicity          |
+------------------+-----------------------------------------------+
| MobileNetV2      | Can bang accuracy/efficiency nho inv-residuals |
|                  | Gradient on dinh hon V1 nho linear bottleneck  |
|                  | --> Khuyen dung cho mobile/edge production     |
+------------------+-----------------------------------------------+
| MobileNetV3-Small| FLOPs thap nhat; h-swish + SE block + NAS      |
|                  | Accuracy/speed trade-off tot nhat trong bo ba  |
|                  | --> Ly tuong cho IoT, real-time inference      |
+------------------+-----------------------------------------------+
| KHI NAO DUNG MOBILENET?                                          |
|   * Thiet bi bien: smartphone, Raspberry Pi, MCU, drone         |
|   * Can latency thap & bo nho RAM/Flash han che                 |
|   * Dataset vua phai, khong can heavy model                     |
|   * Real-time object detection / classification                  |
+------------------------------------------------------------------+
| UU DIEM                                                          |
|   + Nhe – params & FLOPs thap hon ResNet/VGG nhieu lan          |
|   + Transfer learning hieu qua (ImageNet pretrained)            |
|   + De tuy chinh do rong (alpha) & do phan giai                 |
|   + Ho tro tot: TFLite / ONNX / CoreML                          |
| HAN CHE                                                          |
|   x Accuracy kem hon ResNet/EfficientNet tren dataset phuc tap  |
|   x CIFAR-10 (32x32) khong phai diem manh -> can Resizing       |
|   x Depthwise conv it parallel-friendly tren mot so hardware    |
+------------------------------------------------------------------+
''')